In [ ]:
from array import array
from functools import lru_cache
from pathlib import Path
import csv
import gzip
import json
import pickle
import re
import shutil
import time
from typing import Dict, Iterable, List, Optional

import numpy as np
from tqdm.auto import tqdm
from transformers import AutoTokenizer


DATA_ROOT = Path("<DATA_ROOT>")
# Folder with collection.tsv, qidpidtriples.train.full.2.tsv, top1000.dev, qrels.dev.small.tsv, and queries.train.tsv.

DEV_QUERIES_PATH = None
# Optional path to queries.dev.small.tsv; keep None to derive dev queries from top1000.dev.

ARTIFACT_DIR = Path("<ARTIFACT_DIR>")
# Output folder for tokenizer files, dataset caches, and prep_manifest.json.

RUN_PROFILE = 'full'

TOKENIZER_NAME = 'bert-base-uncased'
SEQ_LEN = 256
MAX_QUERY_LEN = 32
MAX_DOC_LEN = 221
RANDOM_SEED = 13
FORCE_REBUILD = False
PASSAGE_TOKEN_SHARD_SIZE = 50_000

REPO_ROOT = Path.cwd().resolve()


def sanitize_tag_fragment(value: str) -> str:
    return re.sub(r'[^A-Za-z0-9._-]+', '-', value).strip('-') or 'tokenizer'


def build_cache_tag(tokenizer_name: str, seq_len: int, max_query_len: int, max_doc_len: int) -> str:
    tokenizer_tag = sanitize_tag_fragment(tokenizer_name)
    return f'{tokenizer_tag}_seq{seq_len}_q{max_query_len}_d{max_doc_len}'


CACHE_TAG = build_cache_tag(TOKENIZER_NAME, SEQ_LEN, MAX_QUERY_LEN, MAX_DOC_LEN)

assert MAX_QUERY_LEN + MAX_DOC_LEN + 3 <= SEQ_LEN


In [ ]:
ARTIFACT_ACTIONS: Dict[str, str] = {}
DATASET_FILENAMES = {
    'collection': 'collection.tsv',
    'train_triples': 'qidpidtriples.train.full.2.tsv',
    'top1000_dev': 'top1000.dev',
    'qrels_dev': 'qrels.dev.small.tsv',
    'train_queries': 'queries.train.tsv',
    'dev_queries': 'queries.dev.small.tsv',
}


def record_artifact_action(path: Path, action: str) -> None:
    ARTIFACT_ACTIONS[str(path)] = action


def config_scoped_artifact_name(stem: str, suffix: str) -> str:
    return f'{stem}_{CACHE_TAG}{suffix}'


DATA_ROOT = Path(DATA_ROOT).expanduser()
ARTIFACT_DIR = Path(ARTIFACT_DIR).expanduser()
DEV_QUERIES_PATH = Path(DEV_QUERIES_PATH).expanduser() if DEV_QUERIES_PATH is not None else None

TOKENIZER_DIR = ARTIFACT_DIR / 'tokenizer'
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATHS = {
    'collection': DATA_ROOT / DATASET_FILENAMES['collection'],
    'train_triples': DATA_ROOT / DATASET_FILENAMES['train_triples'],
    'top1000_dev': DATA_ROOT / DATASET_FILENAMES['top1000_dev'],
    'qrels_dev': DATA_ROOT / DATASET_FILENAMES['qrels_dev'],
    'train_queries': DATA_ROOT / DATASET_FILENAMES['train_queries'],
    'dev_queries': DEV_QUERIES_PATH,
}

DATA_PATHS


In [ ]:
def open_text_auto(path: Path):
    if path.suffix == '.gz':
        return gzip.open(path, 'rt', encoding='utf-8', errors='replace', newline='')
    return path.open('r', encoding='utf-8', errors='replace', newline='')


def save_pickle(obj, path: Path) -> None:
    with path.open('wb') as handle:
        pickle.dump(obj, handle, protocol=pickle.HIGHEST_PROTOCOL)


def load_pickle(path: Path):
    with path.open('rb') as handle:
        return pickle.load(handle)


def maybe_build_pickle(path: Path, builder, force: bool = False):
    if path.exists() and not force:
        print(f'Reusing {path.name}')
        record_artifact_action(path, 'reused')
        return load_pickle(path)
    obj = builder()
    save_pickle(obj, path)
    record_artifact_action(path, 'created')
    return obj


def load_or_save_tokenizer(local_dir: Path, tokenizer_name: str, force: bool = False):
    config_path = local_dir / 'tokenizer_config.json'
    if config_path.exists() and not force:
        print(f'Reusing tokenizer from {local_dir}')
        record_artifact_action(local_dir, 'reused')
        return AutoTokenizer.from_pretrained(local_dir, use_fast=True, local_files_only=True)
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, use_fast=True)
    tokenizer.save_pretrained(local_dir)
    record_artifact_action(local_dir, 'created')
    return tokenizer


tokenizer = load_or_save_tokenizer(TOKENIZER_DIR, TOKENIZER_NAME, force=FORCE_REBUILD)
print({'cls_token_id': tokenizer.cls_token_id, 'sep_token_id': tokenizer.sep_token_id, 'pad_token_id': tokenizer.pad_token_id})


In [ ]:
COLLECTION_OFFSETS_INFO = None


In [ ]:
def load_queries(path: Path) -> Dict[int, str]:
    queries: Dict[int, str] = {}
    with open_text_auto(path) as handle:
        reader = csv.reader(handle, delimiter='\t')
        for row in tqdm(reader, desc=f'Load {path.name}'):
            if not row:
                continue
            qid = int(row[0])
            text = row[1] if len(row) > 1 else ''
            queries[qid] = text
    return queries


def derive_queries_from_top1000(path: Path) -> Dict[int, str]:
    queries: Dict[int, str] = {}
    with open_text_auto(path) as handle:
        reader = csv.reader(handle, delimiter='\t')
        for row in tqdm(reader, desc='Derive dev queries'):
            if len(row) < 3:
                continue
            qid = int(row[0])
            if qid not in queries:
                queries[qid] = row[2]
    return queries


def tokenize_query_map(query_map: Dict[int, str], tokenizer, max_length: int, desc: str) -> Dict[int, List[int]]:
    tokenized: Dict[int, List[int]] = {}
    for qid, text in tqdm(query_map.items(), total=len(query_map), desc=desc):
        tokenized[int(qid)] = tokenizer.encode(
            text,
            add_special_tokens=False,
            truncation=True,
            max_length=max_length,
        )
    return tokenized


train_queries_path = ARTIFACT_DIR / 'train_queries.pkl'
dev_queries_path = ARTIFACT_DIR / 'dev_queries.pkl'
train_query_tokens_path = ARTIFACT_DIR / config_scoped_artifact_name('train_query_tokens', '.pkl')
dev_query_tokens_path = ARTIFACT_DIR / config_scoped_artifact_name('dev_query_tokens', '.pkl')

train_queries = maybe_build_pickle(train_queries_path, lambda: load_queries(DATA_PATHS['train_queries']), force=FORCE_REBUILD)
dev_queries = maybe_build_pickle(
    dev_queries_path,
    lambda: load_queries(DATA_PATHS['dev_queries']) if DATA_PATHS['dev_queries'] is not None else derive_queries_from_top1000(DATA_PATHS['top1000_dev']),
    force=FORCE_REBUILD,
)
train_query_tokens = maybe_build_pickle(
    train_query_tokens_path,
    lambda: tokenize_query_map(train_queries, tokenizer, MAX_QUERY_LEN, desc='Tokenize train queries'),
    force=FORCE_REBUILD,
)
dev_query_tokens = maybe_build_pickle(
    dev_query_tokens_path,
    lambda: tokenize_query_map(dev_queries, tokenizer, MAX_QUERY_LEN, desc='Tokenize dev queries'),
    force=FORCE_REBUILD,
)

print({
    'train_queries': len(train_queries),
    'dev_queries': len(dev_queries),
    'cache_tag': CACHE_TAG,
    'passage_token_shard_size': PASSAGE_TOKEN_SHARD_SIZE,
})


In [ ]:
def resolve_artifact_relative_path(value: str, artifact_dir: Path) -> Path:
    path = Path(value)
    if path.is_absolute():
        return path
    return artifact_dir / path


def write_passage_token_shard(
    output_dir: Path,
    shard_id: int,
    shard_pids: List[int],
    shard_offsets: List[int],
    shard_token_ids: array,
) -> Dict[str, object]:
    if not shard_pids:
        raise ValueError('Cannot write an empty passage token shard')

    pid_start = int(shard_pids[0])
    pid_end = int(shard_pids[-1])
    shard_path = output_dir / f'passage_tokens_{shard_id:05d}_{pid_start:07d}_{pid_end:07d}.npz'
    np.savez_compressed(
        shard_path,
        pid=np.asarray(shard_pids, dtype=np.int32),
        offsets=np.asarray(shard_offsets, dtype=np.int64),
        token_ids=np.asarray(shard_token_ids, dtype=np.int32),
    )
    return {
        'shard_id': int(shard_id),
        'pid_start': pid_start,
        'pid_end': pid_end,
        'pid_count': int(len(shard_pids)),
        'token_count': int(len(shard_token_ids)),
        'path': str(shard_path.relative_to(ARTIFACT_DIR)),
    }


def validate_existing_passage_token_store(index_path: Path) -> bool:
    if not index_path.exists():
        return False
    index = json.loads(index_path.read_text())
    for shard_entry in index.get('shards', []):
        shard_path = resolve_artifact_relative_path(shard_entry['path'], ARTIFACT_DIR)
        if not shard_path.is_file():
            return False
    return True


def build_collection_passage_token_store(
    collection_path: Path,
    output_dir: Path,
    index_path: Path,
    stats_path: Path,
    tokenizer,
    max_doc_len: int,
    shard_size: int,
    force: bool = False,
) -> Dict[str, object]:
    if output_dir.exists() and validate_existing_passage_token_store(index_path) and stats_path.exists() and not force:
        print(f'Reusing {output_dir.name}')
        record_artifact_action(output_dir, 'reused')
        record_artifact_action(index_path, 'reused')
        record_artifact_action(stats_path, 'reused')
        return json.loads(stats_path.read_text())

    if force and output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    shards: List[Dict[str, object]] = []
    last_pid = -1
    current_shard_id: Optional[int] = None
    current_pids: List[int] = []
    current_offsets: List[int] = [0]
    current_token_ids = array('I')

    total_bytes = collection_path.stat().st_size
    with collection_path.open('rb') as handle, tqdm(total=total_bytes, unit='B', unit_scale=True, desc='Tokenize collection passages') as pbar:
        for raw_line in handle:
            if not raw_line:
                continue
            pbar.update(len(raw_line))
            row = raw_line.decode('utf-8', errors='replace').rstrip(chr(10))
            if '\t' not in row:
                continue
            raw_pid, passage_text = row.split('\t', 1)
            pid = int(raw_pid)
            if pid < last_pid:
                raise ValueError('collection.tsv must be sorted by pid for pid-range sharding')
            last_pid = pid
            shard_id = pid // shard_size
            if current_shard_id is None:
                current_shard_id = shard_id
            elif shard_id != current_shard_id:
                shards.append(write_passage_token_shard(output_dir, current_shard_id, current_pids, current_offsets, current_token_ids))
                current_shard_id = shard_id
                current_pids = []
                current_offsets = [0]
                current_token_ids = array('I')

            token_ids = tokenizer.encode(
                passage_text,
                add_special_tokens=False,
                truncation=True,
                max_length=max_doc_len,
            )
            current_pids.append(pid)
            current_token_ids.extend(int(token_id) for token_id in token_ids)
            current_offsets.append(len(current_token_ids))

    if current_shard_id is not None and current_pids:
        shards.append(write_passage_token_shard(output_dir, current_shard_id, current_pids, current_offsets, current_token_ids))

    index_payload = {
        'format': 'sharded_flat_token_arrays_v1',
        'created_at_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'tokenizer_name': TOKENIZER_NAME,
        'cache_tag': CACHE_TAG,
        'max_doc_len': int(max_doc_len),
        'shard_size': int(shard_size),
        'shard_count': int(len(shards)),
        'shards': shards,
    }
    stats_payload = {
        'format': index_payload['format'],
        'created_at_utc': index_payload['created_at_utc'],
        'cache_tag': CACHE_TAG,
        'output_dir': str(output_dir.relative_to(ARTIFACT_DIR)),
        'index_artifact': index_path.name,
        'stats_artifact': stats_path.name,
        'tokenizer_name': TOKENIZER_NAME,
        'max_doc_len': int(max_doc_len),
        'shard_size': int(shard_size),
        'shard_count': int(len(shards)),
        'passage_count': int(sum(int(entry['pid_count']) for entry in shards)),
        'token_count': int(sum(int(entry['token_count']) for entry in shards)),
        'max_pid': int(max((int(entry['pid_end']) for entry in shards), default=-1)),
    }

    index_path.write_text(json.dumps(index_payload, indent=2))
    stats_path.write_text(json.dumps(stats_payload, indent=2))
    record_artifact_action(output_dir, 'created')
    record_artifact_action(index_path, 'created')
    record_artifact_action(stats_path, 'created')
    return stats_payload


def load_passage_token_shard_index(index_path: Path) -> Dict[str, object]:
    return json.loads(index_path.read_text())


def make_passage_token_getter(index_path: Path, artifact_dir: Path, shard_cache_size: int = 4):
    index = load_passage_token_shard_index(index_path)
    if index.get('format') != 'sharded_flat_token_arrays_v1':
        raise ValueError(f"Unsupported passage token store format: {index.get('format')!r}")

    shard_size = int(index['shard_size'])
    shards_by_id = {int(entry['shard_id']): entry for entry in index['shards']}

    @lru_cache(maxsize=shard_cache_size)
    def load_shard(shard_id: int) -> Dict[str, object]:
        shard_entry = shards_by_id.get(int(shard_id))
        if shard_entry is None:
            raise KeyError(f'No passage token shard found for shard_id={shard_id}')
        shard_path = resolve_artifact_relative_path(shard_entry['path'], artifact_dir)
        with np.load(shard_path, allow_pickle=False) as shard_data:
            shard_pids = shard_data['pid']
            shard_offsets = shard_data['offsets']
            shard_token_ids = shard_data['token_ids']
        pid_lookup = {int(pid): idx for idx, pid in enumerate(shard_pids.tolist())}
        return {
            'pid': shard_pids,
            'offsets': shard_offsets,
            'token_ids': shard_token_ids,
            'pid_lookup': pid_lookup,
        }

    def get_passage_tokens(pid: int) -> List[int]:
        shard = load_shard(int(pid) // shard_size)
        pid_idx = shard['pid_lookup'].get(int(pid))
        if pid_idx is None:
            raise KeyError(f'Missing cached passage tokens for pid={pid}')
        start = int(shard['offsets'][pid_idx])
        end = int(shard['offsets'][pid_idx + 1])
        return shard['token_ids'][start:end].tolist()

    return get_passage_tokens


def load_passage_tokens_for_subset(pid_list: Iterable[int], index_path: Path, artifact_dir: Path) -> Dict[int, List[int]]:
    get_passage_tokens = make_passage_token_getter(index_path, artifact_dir)
    unique_pids = sorted({int(pid) for pid in pid_list})
    return {
        pid: get_passage_tokens(pid)
        for pid in tqdm(unique_pids, total=len(unique_pids), desc='Load cached passage subset')
    }


passage_token_shards_dir = ARTIFACT_DIR / config_scoped_artifact_name('passage_token_shards', '')
passage_token_shards_index_path = ARTIFACT_DIR / config_scoped_artifact_name('passage_token_shards_index', '.json')
passage_token_store_stats_path = ARTIFACT_DIR / config_scoped_artifact_name('passage_token_store_stats', '.json')
PASSAGE_TOKEN_STORE_INFO = build_collection_passage_token_store(
    DATA_PATHS['collection'],
    passage_token_shards_dir,
    passage_token_shards_index_path,
    passage_token_store_stats_path,
    tokenizer,
    MAX_DOC_LEN,
    shard_size=PASSAGE_TOKEN_SHARD_SIZE,
    force=FORCE_REBUILD,
)

print(PASSAGE_TOKEN_STORE_INFO)


In [ ]:
def load_dev_qrels(path: Path) -> Dict[int, set]:
    qrels: Dict[int, set] = {}
    with open_text_auto(path) as handle:
        reader = csv.reader(handle, delimiter='\t')
        for row in reader:
            if len(row) < 4:
                continue
            qid = int(row[0])
            pid = int(row[2])
            rel = int(row[3])
            if rel > 0:
                qrels.setdefault(qid, set()).add(pid)
    return qrels


def parse_dev_candidates(top1000_path: Path, output_path: Path, force: bool = False):
    if output_path.exists() and not force:
        print(f'Reusing {output_path.name}')
        record_artifact_action(output_path, 'reused')
        return load_pickle(output_path)

    qid_order: List[int] = []
    qid_to_pids: Dict[int, List[int]] = {}

    with open_text_auto(top1000_path) as handle:
        reader = csv.reader(handle, delimiter='\t')
        for row in tqdm(reader, desc='Parse dev candidates'):
            if len(row) < 2:
                continue
            qid = int(row[0])
            pid = int(row[1])
            if qid not in qid_to_pids:
                qid_to_pids[qid] = []
                qid_order.append(qid)
            qid_to_pids[qid].append(pid)

    qid_offsets: List[int] = [0]
    pid_values: List[int] = []
    rank_values: List[int] = []
    for qid in qid_order:
        pids = qid_to_pids[qid]
        pid_values.extend(pids)
        rank_values.extend(range(1, len(pids) + 1))
        qid_offsets.append(len(pid_values))

    artifact = {
        'format': 'grouped_arrays_v1',
        'qid_order': np.asarray(qid_order, dtype=np.int32),
        'qid_offsets': np.asarray(qid_offsets, dtype=np.int64),
        'pid': np.asarray(pid_values, dtype=np.int32),
        'bm25_rank': np.asarray(rank_values, dtype=np.int16),
    }
    save_pickle(artifact, output_path)
    record_artifact_action(output_path, 'created')
    return artifact



dev_candidates_path = ARTIFACT_DIR / 'dev_candidates.pkl'
dev_qrels_path = ARTIFACT_DIR / 'dev_qrels.pkl'

dev_candidates = parse_dev_candidates(DATA_PATHS['top1000_dev'], dev_candidates_path, force=FORCE_REBUILD)
dev_qrels = maybe_build_pickle(dev_qrels_path, lambda: load_dev_qrels(DATA_PATHS['qrels_dev']), force=FORCE_REBUILD)
print({
    'dev_queries_with_candidates': len(dev_candidates['qid_order']),
    'dev_candidate_rows': int(len(dev_candidates['pid'])),
    'dev_qrels_queries': len(dev_qrels),
})


In [ ]:
def write_json_with_action(path: Path, payload: Dict[str, object]) -> None:
    action = 'updated' if path.exists() else 'created'
    path.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    record_artifact_action(path, action)


def build_artifact_index(artifact_dir: Path, index_path: Path, artifacts: Dict[str, str]) -> Dict[str, object]:
    files = {}
    for label in sorted(set(artifacts.values())):
        artifact_path = artifact_dir / label
        if artifact_path.is_file():
            files[label] = {'type': 'file', 'size_bytes': artifact_path.stat().st_size}
        elif artifact_path.is_dir():
            files[label] = {'type': 'directory'}

    artifact_index = {
        'created_at_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'artifact_dir': str(artifact_dir.resolve()),
        'files': files,
    }
    index_path.write_text(json.dumps(artifact_index, indent=2), encoding='utf-8')
    artifact_index['files'][index_path.name] = {'type': 'file', 'size_bytes': index_path.stat().st_size}
    index_path.write_text(json.dumps(artifact_index, indent=2), encoding='utf-8')
    return artifact_index


def build_manifest_artifacts() -> Dict[str, str]:
    return {
        'train_queries_pkl': train_queries_path.name,
        'dev_queries_pkl': dev_queries_path.name,
        'train_query_tokens_pkl': train_query_tokens_path.name,
        'dev_query_tokens_pkl': dev_query_tokens_path.name,
        'passage_token_shards_dir': passage_token_shards_dir.name,
        'passage_token_shards_index_json': passage_token_shards_index_path.name,
        'passage_token_store_stats_json': passage_token_store_stats_path.name,
        'dev_candidates_pkl': dev_candidates_path.name,
        'dev_qrels_pkl': dev_qrels_path.name,
        'cache_manifest_json': cache_manifest_path.name,
        'prep_manifest_json': prep_manifest_path.name,
        'artifact_index_json': artifact_index_path.name,
    }


cache_manifest_path = ARTIFACT_DIR / f'cache_manifest_{CACHE_TAG}.json'
prep_manifest_path = ARTIFACT_DIR / 'prep_manifest.json'
artifact_index_path = ARTIFACT_DIR / 'artifact_index.json'
artifact_index_existed = artifact_index_path.exists()
manifest_artifacts = build_manifest_artifacts()

manifest = {
    'schema_version': 4,
    'created_at_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'path_config': 'explicit',
    'repo_root_at_creation': str(REPO_ROOT),
    'artifact_dir_at_creation': str(ARTIFACT_DIR.resolve()),
    'data_root': str(DATA_ROOT),
    'run_profile': RUN_PROFILE,
    'dataset_cache_scope': 'dataset_level',
    'cache_tag': CACHE_TAG,
    'compatibility_keys': ['tokenizer_name', 'seq_len', 'max_query_len', 'max_doc_len'],
    'tokenizer_name': TOKENIZER_NAME,
    'tokenizer_local_path': 'tokenizer',
    'seq_len': SEQ_LEN,
    'max_query_len': MAX_QUERY_LEN,
    'max_doc_len': MAX_DOC_LEN,
    'passage_token_store_format': PASSAGE_TOKEN_STORE_INFO['format'],
    'passage_token_shard_size': PASSAGE_TOKEN_STORE_INFO['shard_size'],
    'random_seed': RANDOM_SEED,
    'dataset_paths': {key: (str(value.resolve()) if value is not None else None) for key, value in DATA_PATHS.items()},
    'dataset_filenames': {key: (value.name if value is not None else None) for key, value in DATA_PATHS.items()},
    'artifacts': manifest_artifacts,
    'created_files': sorted(manifest_artifacts.values()),
    'collection_offsets_info': COLLECTION_OFFSETS_INFO,
    'passage_token_store_info': PASSAGE_TOKEN_STORE_INFO,
    'dev_candidates_format': 'grouped_arrays_v1',
}
write_json_with_action(cache_manifest_path, manifest)
write_json_with_action(prep_manifest_path, manifest)
artifact_index = build_artifact_index(ARTIFACT_DIR, artifact_index_path, manifest_artifacts)
record_artifact_action(artifact_index_path, 'updated' if artifact_index_existed else 'created')
manifest


In [ ]:
def summarize_artifact_actions(actions: Dict[str, str]) -> Dict[str, str]:
    summary = {}
    for raw_path, action in sorted(actions.items()):
        path = Path(raw_path)
        try:
            label = str(path.relative_to(ARTIFACT_DIR))
        except ValueError:
            label = str(path)
        summary[label] = action
    return summary


summary = {
    'path_config': 'explicit',
    'run_profile': RUN_PROFILE,
    'cache_tag': CACHE_TAG,
    'dataset_cache_scope': 'dataset_level',
    'data_root': str(DATA_ROOT),
    'artifact_dir': str(ARTIFACT_DIR.resolve()),
    'prep_manifest_path': str(prep_manifest_path),
    'cache_manifest_path': str(cache_manifest_path),
    'resolved_dataset_paths': {key: (str(value) if value is not None else None) for key, value in DATA_PATHS.items()},
    'created_or_reused_artifacts': summarize_artifact_actions(ARTIFACT_ACTIONS),
    'next_notebook': '01_prepare_run_data_cache.ipynb',
}
print(json.dumps(summary, indent=2))
print('Next: run 01_prepare_run_data_cache.ipynb, then train_main.ipynb')
summary
